# 7.4 · B-Trees — Challenges

In this lab, we will work on the main operations of a **B-tree**: searching, inserting, and deleting keys. At the end, we will also solve diagram-based exercises.


## 📌 Conventions

We will mainly work with **B-trees of order 4**:

- at most **4 children** per node;
- at most **3 keys** per node;
- every non-root node must contain at least **1 key** (the tree may have an empty root only as an intermediate state).

We represent a node by placing its keys inside brackets:

```text
[10 | 20]
```

and its children underneath:

```text
          [10 | 20]
        /     |      \
      [5]   [15]   [25 | 30]
```


## How to work through this notebook

First, run the cell containing the classes. Complete each challenge by **filling in the `TODO` sections in that same cell**, then rerun it before running the corresponding tests. Programming challenges build on previous ones; diagram-based exercises do not require code.

Duplicate keys are not allowed.


## Starter code for the `BTreeNode` and `BTree` classes


In [ ]:
class BTreeNode:
    def __init__(self, is_leaf=True):
        # TODO (Challenge 1)
        pass

    def insert_key(self, key):
        # TODO (Challenge 2)
        pass


class BTree:
    def __init__(self, m):
        # TODO (Challenge 1)
        pass

    def search(self, key):
        # TODO (Challenge 3)
        pass

    def _search_recursive(self, node, key):
        # TODO (Challenge 3)
        pass

    def search_leaf(self, key):
        # TODO (Challenge 4)
        pass

    def insert(self, key):
        # TODO (Challenge 5)
        pass

    def _split(self, node):
        # TODO (Challenge 5)
        pass

    def _get_predecessor(self, node, i):
        # TODO (Challenge 6)
        pass

    def _borrow_from_left(self, node):
        # TODO (Challenge 7)
        pass

    def _borrow_from_right(self, node):
        # TODO (Challenge 7)
        pass

    def _merge(self, node):
        # TODO (Challenge 8)
        pass

    def _fix_underflow(self, node):
        # TODO (Challenge 9)
        pass

    def remove(self, key):
        # TODO (Challenge 10)
        pass


# 🏆 Challenge 1 · Implement the `BTreeNode` and `BTree` constructors

Each node in a B-tree of order `m` stores sorted keys and references to child nodes. The `BTreeNode` class has these attributes:

- **`keys`:** a Python list of keys in ascending order. It holds at most **m − 1 keys**.
- **`children`:** a list of references to child nodes. Internal nodes have at most **m children**, whereas a leaf has none.
- **`parent`:** a reference to the parent node, or `None` for the root.
- **`is_leaf`:** `True` for leaf nodes and `False` for internal nodes. This supports searching, insertion, and deletion.

The `BTree` class represents the entire B-tree and has these attributes:

- **`root`:** a reference to the root node; initially `None` because the tree starts empty.
- **`m`:** the order of the B-tree, determining the maximum number of children (**m**) and keys (**m − 1**) per node.


Run the following cell to check your solution.


In [ ]:
# Tests Challenge 1: BTreeNode and BTree constructors

# Test 1: Create a leaf node
node = BTreeNode()
assert node.keys == [], "keys debe ser una lista vacía"
assert node.children == [], "children debe ser una lista vacía"
assert node.parent is None, "parent debe ser None"
assert node.is_leaf is True, "Por defecto, el nodo debe ser una hoja"

# Test 2: Create an internal node
node = BTreeNode(is_leaf=False)
assert node.is_leaf is False, "El nodo debe ser interno"
assert node.keys == [], "keys debe estar vacía"
assert node.children == [], "children debe estar vacía"
assert node.parent is None, "parent debe ser None"

# Test 3: Create an order-4 B-tree
tree = BTree(4)
assert tree.m == 4, "El orden del árbol debe ser 4"
assert tree.root is None, "La raíz debe ser None inicialmente"

# Test 4: Create a B-tree of a different order
tree = BTree(6)
assert tree.m == 6, "El orden del árbol debe ser 6"
assert tree.root is None, "El árbol debe estar vacío inicialmente"

# Test 5: Check that nodes have independent lists
node1 = BTreeNode()
node2 = BTreeNode()
assert node1.keys is not node2.keys, "Cada nodo debe tener su propia lista de claves"
assert node1.children is not node2.children, "Cada nodo debe tener su propia lista de hijos"

print("Reto 1: Todos los tests superados correctamente.")


# 🏆 Challenge 2 · Insert a key into a node

**Goal:** Implement `BTreeNode.insert_key(key)` to add a key to `self.keys` while keeping keys **in ascending order**.

- If the key is not present, insert it in the correct position and return `True`.
- If the key already exists, do not modify the list and return `False`.

**Important:** This method only updates the sorted list. **It does not check whether the node exceeds the maximum number of keys or perform splits.** We will handle these operations later.

**Estimated time:** 8 minutes · **Difficulty:** Easy

**Hint:** Scan the list until you find the correct insertion position. You can use Python's `list.insert(pos, value)`.


In [ ]:
# Tests Challenge 2: BTreeNode.insert_key()

node = BTreeNode()

# Test 1: Insert into an empty node
assert node.insert_key(20) is True
assert node.keys == [20]

# Test 2: Insert a smaller key
assert node.insert_key(10) is True
assert node.keys == [10, 20]

# Test 3: Insert a larger key
assert node.insert_key(30) is True
assert node.keys == [10, 20, 30]

# Test 4: Insert a key in the middle
assert node.insert_key(15) is True
assert node.keys == [10, 15, 20, 30]

# Test 5: Try inserting a duplicate key
assert node.insert_key(20) is False
assert node.keys == [10, 15, 20, 30]

# Test 6: Insert a negative key
assert node.insert_key(-5) is True
assert node.keys == [-5, 10, 15, 20, 30]

print("Reto 2: Todos los tests superados correctamente.")

# 🏆 Challenge 3 · Search for a key in a B-tree

**Goal:** Implement `BTree.search(key)` and `BTree._search_recursive(node, key)` using recursive search.

- **`search(key)`:** public method that starts at the root.
- **`_search_recursive(node, key)`:** helper method that recursively searches from a node.

**Return value:** If the key is found, return `(node, i)`, where `node` contains the key and `i` is its index in `node.keys`. Otherwise, return `None`.

**Important:** Searching must not modify the keys or the tree structure.

**Estimated time:** 15 minutes · **Difficulty:** Easy

### Hint

At each node:

1. Find the first index `i` whose key is greater than or equal to `key`.
2. If `node.keys[i] == key`, return `(node, i)`.
3. If there is no match and the node is a leaf, return `None`.
4. Otherwise, recursively search `node.children[i]`.

**Remember:** Make sure `i` is a valid index before accessing `node.keys[i]`.


Since we have not implemented `insert()` yet, we will manually construct an **order-4 B-tree** to check that `search()` returns `(node, i)` when the key exists and `None` otherwise.

**Run the following code cell** to check your implementation.


In [ ]:
# Tests Challenge 3: BTree.search()

# Create an order-4 B-tree
tree = BTree(4)

# Create the nodes
root = BTreeNode(is_leaf=False)
left = BTreeNode(is_leaf=False)
right = BTreeNode(is_leaf=False)

# Assign the keys
root.keys = [20]
left.keys = [10, 15]
right.keys = [35]

# Create the leaves
leaf1 = BTreeNode()
leaf1.keys = [5, 8]

leaf2 = BTreeNode()
leaf2.keys = [12]

leaf3 = BTreeNode()
leaf3.keys = [16, 18]

leaf4 = BTreeNode()
leaf4.keys = [22, 30]

leaf5 = BTreeNode()
leaf5.keys = [36]

# Set up the links between nodes
tree.root = root
root.children = [left, right]
left.children = [leaf1, leaf2, leaf3]
right.children = [leaf4, leaf5]

# Assign parent references
left.parent = root
right.parent = root

for leaf in [leaf1, leaf2, leaf3]:
    leaf.parent = left

for leaf in [leaf4, leaf5]:
    leaf.parent = right


# Test 1: Search for a key in the root
assert tree.search(20) == (root, 0)

# Test 2: Search for keys in internal nodes
assert tree.search(10) == (left, 0)
assert tree.search(15) == (left, 1)
assert tree.search(35) == (right, 0)

# Test 3: Search for keys in leaves
assert tree.search(5) == (leaf1, 0)
assert tree.search(8) == (leaf1, 1)
assert tree.search(12) == (leaf2, 0)
assert tree.search(18) == (leaf3, 1)
assert tree.search(30) == (leaf4, 1)
assert tree.search(36) == (leaf5, 0)

# Test 4: Search for missing keys
assert tree.search(7) is None
assert tree.search(14) is None
assert tree.search(25) is None
assert tree.search(40) is None

# Test 5: Search an empty tree
empty_tree = BTree(4)
assert empty_tree.search(10) is None

print("Reto 3: Todos los tests superados correctamente.")

# 🏆 Challenge 4 · Find the leaf where a key should be inserted

🎯 **Goal:** Implement `BTree.search_leaf(key)` to return the **leaf node into which `key` should be inserted**.

- If the key already exists, return `None`.
- If the tree is empty, return `None`.

**Important:** Only locate the leaf; do not insert the key or modify the tree.

🕒 **Estimated time:** 10 minutes  
🟢 **Difficulty:** Easy

### 💡 Hint

This is similar to `search()`: at each node, check whether the key exists and, if it does not, follow the appropriate child until reaching a leaf.


In [ ]:
# Tests Challenge 4: BTree.search_leaf()

# Use the 'tree' built in Challenge 3

# Test 1: Find leaves for new keys
assert tree.search_leaf(3) is leaf1
assert tree.search_leaf(7) is leaf1
assert tree.search_leaf(11) is leaf2
assert tree.search_leaf(14) is leaf2
assert tree.search_leaf(17) is leaf3
assert tree.search_leaf(25) is leaf4
assert tree.search_leaf(40) is leaf5

# Test 2: Keys already present in leaves
assert tree.search_leaf(5) is None
assert tree.search_leaf(12) is None
assert tree.search_leaf(18) is None
assert tree.search_leaf(36) is None

# Test 3: Keys already present in internal nodes
assert tree.search_leaf(10) is None
assert tree.search_leaf(15) is None
assert tree.search_leaf(20) is None
assert tree.search_leaf(35) is None

# Test 4: Empty tree
empty_tree = BTree(4)
assert empty_tree.search_leaf(10) is None

# Test 5: Check that searching does not modify the tree
original_keys = leaf2.keys.copy()
assert tree.search_leaf(13) is leaf2
assert leaf2.keys == original_keys

print("Reto 4: Todos los tests superados correctamente.")


# 🏆 Challenge 5 · Insert a key into a B-tree

🎯 **Goal:** Implement `BTree.insert(key)` and its helper method `_split(node)` to insert keys while preserving B-tree properties.

**`insert(key)` must:**

1. Create the root if the tree is empty.
2. Prevent duplicate keys.
3. Locate the leaf using `search_leaf()` and insert the key using `insert_key()`.
4. Split any node that exceeds `m - 1` keys, propagating splits upward as needed.

Return `True` if the key was inserted, or `False` if it already exists.

**`_split(node)` must:**

1. Split an overflowing node into two nodes, distributing its keys and children.
2. Promote the middle key to the parent. If there are two middle keys, choose the right-hand one (the third key in a four-key node).
3. Create a new root if splitting the root.
4. Return the parent so `insert()` can check whether it also needs splitting. If a new root was created, return `None`.

**Important:** `_split(node)` performs just one split; `insert()` propagates splits using a `while` loop.

🕒 **Estimated time:** 25 minutes  
🟡 **Difficulty:** Intermediate

### 💡 Hint

Use `len(node.keys) // 2` to select the promoted key. Remember to update `parent` references and distribute child references when splitting an internal node.


In [ ]:
# Tests Challenge 5: BTree.insert() and BTree._split()

tree = BTree(4)

# Test 1: Insert into an empty tree
assert tree.insert(5) is True
assert tree.root.keys == [5]
assert tree.root.is_leaf is True

# Test 2: Insert without splitting
assert tree.insert(10) is True
assert tree.insert(15) is True
assert tree.root.keys == [5, 10, 15]

# Test 3: Split the root
assert tree.insert(16) is True
assert tree.root.keys == [15]
assert tree.root.is_leaf is False
assert tree.root.children[0].keys == [5, 10]
assert tree.root.children[1].keys == [16]

# Test 4: Insert more keys
for key in [8, 12, 18, 20, 22, 30, 35]:
    assert tree.insert(key) is True

# Test 5: Split propagating to the root
assert tree.insert(36) is True

assert tree.root.keys == [20]
assert tree.root.children[0].keys == [10, 15]
assert tree.root.children[1].keys == [35]

# Test 6: Check the leaves
left = tree.root.children[0]
right = tree.root.children[1]

assert [child.keys for child in left.children] == [
    [5, 8], [12], [16, 18]
]

assert [child.keys for child in right.children] == [
    [22, 30], [36]
]

# Test 7: Check parent references
assert left.parent is tree.root
assert right.parent is tree.root

for child in left.children:
    assert child.parent is left

for child in right.children:
    assert child.parent is right

# Test 8: Prevent duplicate keys
assert tree.insert(20) is False
assert tree.insert(12) is False
assert tree.insert(36) is False

# Test 9: Check that all keys can be found
for key in [5, 10, 15, 16, 8, 12, 18, 20, 22, 30, 35, 36]:
    assert tree.search(key) is not None

# Test 10: Check that missing keys cannot be found
assert tree.search(7) is None
assert tree.search(25) is None
assert tree.search(40) is None

print("Reto 5: Todos los tests superados correctamente.")

# 🏆 Challenge 6 · Find the predecessor of a key

🎯 **Goal:** Implement `BTree._get_predecessor(node, i)` to return the **predecessor of a key in an internal node**.

The predecessor of `node.keys[i]` is the largest key in its left subtree.

Return `(leaf, key)`, where:

- `leaf` is the leaf node containing the predecessor.
- `key` is the predecessor value.

**Important:** Only locate the predecessor; do not delete keys or modify the tree.

🕒 **Estimated time:** 10 minutes  
🟢 **Difficulty:** Easy

### 💡 Hint

Start at `node.children[i]` and repeatedly follow the last child until you reach a leaf. Its last key is the predecessor.


Run the following cell to check your `_get_predecessor()` implementation.

Although we have already implemented `insert()`, we will build the test tree manually so the tests do not depend on whether your insertion method is correct.


In [ ]:
# Tests Challenge 6: BTree._get_predecessor()

# Create an order-4 B-tree
tree = BTree(4)

# Create the root
root = BTreeNode(is_leaf=False)
root.keys = [20]
tree.root = root

# Create the internal nodes
left = BTreeNode(is_leaf=False)
left.keys = [10, 15]

right = BTreeNode(is_leaf=False)
right.keys = [35]

root.children = [left, right]
left.parent = root
right.parent = root

# Create the leaves
leaf1 = BTreeNode()
leaf1.keys = [5, 8]

leaf2 = BTreeNode()
leaf2.keys = [12]

leaf3 = BTreeNode()
leaf3.keys = [16, 18]

leaf4 = BTreeNode()
leaf4.keys = [22, 30]

leaf5 = BTreeNode()
leaf5.keys = [36]

# Set up parent-child references
left.children = [leaf1, leaf2, leaf3]
right.children = [leaf4, leaf5]

for leaf in left.children:
    leaf.parent = left

for leaf in right.children:
    leaf.parent = right


# Test 1: Predecessor of root key (20)
leaf, key = tree._get_predecessor(root, 0)
assert leaf is leaf3
assert key == 18

# Test 2: Predecessor of key 10
leaf, key = tree._get_predecessor(left, 0)
assert leaf is leaf1
assert key == 8

# Test 3: Predecessor of key 15
leaf, key = tree._get_predecessor(left, 1)
assert leaf is leaf2
assert key == 12

# Test 4: Predecessor of key 35
leaf, key = tree._get_predecessor(right, 0)
assert leaf is leaf4
assert key == 30

# Test 5: Check that the tree is unchanged
original_keys = leaf4.keys.copy()
tree._get_predecessor(right, 0)

assert leaf4.keys == original_keys
assert tree.root.keys == [20]

print("Reto 6: Todos los tests superados correctamente.")

# 🏆 Challenge 7 · Redistribute keys between sibling nodes

🎯 **Goal:** Implement `BTree._borrow_from_left(node)` and `BTree._borrow_from_right(node)` to fix a node with fewer than the required minimum keys by **borrowing from a sibling**.

- **`_borrow_from_left(node)`:** borrow a key from the left sibling through the parent.
- **`_borrow_from_right(node)`:** borrow a key from the right sibling through the parent.

Both methods must:

1. Redistribute keys among the sibling, parent, and deficient node in sorted order.
2. If the nodes are internal, transfer the appropriate child and update its `parent` reference.

**Important:** The sibling must have the same parent and more than the minimum number of keys. Each method performs only one redistribution and returns nothing.

🕒 **Estimated time:** 20 minutes  
🟡 **Difficulty:** Intermediate

### 💡 Hint

- **Borrowing from the left:** the sibling's largest key moves up to the parent, and the parent's separating key moves down to the beginning of `node.keys`.
- **Borrowing from the right:** the sibling's smallest key moves up to the parent, and the parent's separating key is appended to `node.keys`.

You can use Python list methods `pop()`, `insert()`, and `append()`.


This cell manually constructs several order-4 B-trees to test borrowing from the left and right siblings, for both leaf and internal nodes.

The tests do not depend on `insert()` or `remove()`.


In [ ]:
# Tests Challenge 7: Redistribution between siblings

# Test 1: Borrow from the left sibling (leaves)
tree = BTree(4)

root = BTreeNode(is_leaf=False)
root.keys = [20]
tree.root = root

left = BTreeNode()
left.keys = [5, 10]

right = BTreeNode()
right.keys = []  # Nodo deficitario

root.children = [left, right]
left.parent = root
right.parent = root

tree._borrow_from_left(right)

assert root.keys == [10]
assert left.keys == [5]
assert right.keys == [20]
assert root.children == [left, right]


# Test 2: Borrow from the right sibling (leaves)
tree = BTree(4)

root = BTreeNode(is_leaf=False)
root.keys = [20]
tree.root = root

left = BTreeNode()
left.keys = []  # Nodo deficitario

right = BTreeNode()
right.keys = [30, 40]

root.children = [left, right]
left.parent = root
right.parent = root

tree._borrow_from_right(left)

assert root.keys == [30]
assert left.keys == [20]
assert right.keys == [40]
assert root.children == [left, right]


# Test 3: Borrow from the left sibling (internal nodes)
tree = BTree(4)

root = BTreeNode(is_leaf=False)
root.keys = [30]
tree.root = root

left = BTreeNode(is_leaf=False)
left.keys = [10, 20]

right = BTreeNode(is_leaf=False)
right.keys = []  # Nodo deficitario

root.children = [left, right]
left.parent = root
right.parent = root

# Create the leaves
a = BTreeNode()
a.keys = [5]

b = BTreeNode()
b.keys = [15]

c = BTreeNode()
c.keys = [25]

d = BTreeNode()
d.keys = [40]

left.children = [a, b, c]
right.children = [d]

for child in left.children:
    child.parent = left
d.parent = right

tree._borrow_from_left(right)

assert root.keys == [20]
assert left.keys == [10]
assert right.keys == [30]

assert left.children == [a, b]
assert right.children == [c, d]
assert c.parent is right
assert d.parent is right


# Test 4: Borrow from the right sibling (internal nodes)
tree = BTree(4)

root = BTreeNode(is_leaf=False)
root.keys = [20]
tree.root = root

left = BTreeNode(is_leaf=False)
left.keys = []  # Nodo deficitario

right = BTreeNode(is_leaf=False)
right.keys = [30, 40]

root.children = [left, right]
left.parent = root
right.parent = root

# Create the leaves
a = BTreeNode()
a.keys = [10]

b = BTreeNode()
b.keys = [25]

c = BTreeNode()
c.keys = [35]

d = BTreeNode()
d.keys = [45]

left.children = [a]
right.children = [b, c, d]

a.parent = left
for child in right.children:
    child.parent = right

tree._borrow_from_right(left)

assert root.keys == [30]
assert left.keys == [20]
assert right.keys == [40]

assert left.children == [a, b]
assert right.children == [c, d]
assert a.parent is left
assert b.parent is left
assert c.parent is right
assert d.parent is right

print("Reto 7: Todos los tests superados correctamente.")

# 🏆 Challenge 8 · Merge sibling nodes

🎯 **Goal:** Implement `BTree._merge(node)` to fix a node with too few keys by **merging it with a sibling and a separating key from its parent**.

The method must:

1. **Select an adjacent sibling**, preferring the left sibling if available.
2. **Merge the keys** from both nodes with the parent separator in sorted order.
3. If the nodes are internal, **combine their children** and update `parent` references.
4. **Update the parent**, removing the separator and the reference to the node that disappears.
5. **Return the parent** so we can later check whether it is also deficient.

**Important:** This method performs only one merge. It does not check or propagate a deficiency in the parent; we will do that in the next challenge.

🕒 **Estimated time:** 25 minutes  
🟠 **Difficulty:** Intermediate–advanced

### 💡 Hint

Use `parent.children.index(node)` to determine the node's position among its siblings.

Remember:

- When merging with the **left sibling**, the parent separator is `parent.keys[i - 1]`.
- When merging with the **right sibling**, the parent separator is `parent.keys[i]`.

You can use the Python list methods `pop()`, `append()`, and `extend()`.


This cell tests Challenge 8, including merges with left and right siblings, for both leaves and internal nodes.

The trees are built manually so the tests do not depend on `insert()`.


In [ ]:
# Tests Challenge 8: BTree._merge()

# Test 1: Merge with left sibling (leaves)
tree = BTree(4)

root = BTreeNode(is_leaf=False)
root.keys = [10, 20]
tree.root = root

left = BTreeNode()
left.keys = [5]

node = BTreeNode()
node.keys = []  # Nodo deficitario

right = BTreeNode()
right.keys = [25]

root.children = [left, node, right]
for child in root.children:
    child.parent = root

parent = tree._merge(node)

assert parent is root
assert root.keys == [20]
assert root.children == [left, right]
assert left.keys == [5, 10]
assert left.parent is root


# Test 2: Merge with right sibling (leaves)
tree = BTree(4)

root = BTreeNode(is_leaf=False)
root.keys = [20]
tree.root = root

node = BTreeNode()
node.keys = []  # Nodo deficitario

right = BTreeNode()
right.keys = [30]

root.children = [node, right]
node.parent = root
right.parent = root

parent = tree._merge(node)

assert parent is root
assert root.keys == []
assert root.children == [node]
assert node.keys == [20, 30]
assert node.parent is root


# Test 3: Merge with left sibling (internal nodes)
tree = BTree(4)

root = BTreeNode(is_leaf=False)
root.keys = [30]
tree.root = root

left = BTreeNode(is_leaf=False)
left.keys = [10]

node = BTreeNode(is_leaf=False)
node.keys = []  # Nodo deficitario

root.children = [left, node]
left.parent = root
node.parent = root

# Create the leaves
a = BTreeNode()
a.keys = [5]

b = BTreeNode()
b.keys = [20]

c = BTreeNode()
c.keys = [40]

left.children = [a, b]
node.children = [c]

for child in left.children:
    child.parent = left
c.parent = node

parent = tree._merge(node)

assert parent is root
assert root.keys == []
assert root.children == [left]
assert left.keys == [10, 30]
assert left.children == [a, b, c]

for child in left.children:
    assert child.parent is left


# Test 4: Merge with right sibling (internal nodes)
tree = BTree(4)

root = BTreeNode(is_leaf=False)
root.keys = [20]
tree.root = root

node = BTreeNode(is_leaf=False)
node.keys = []  # Nodo deficitario

right = BTreeNode(is_leaf=False)
right.keys = [30]

root.children = [node, right]
node.parent = root
right.parent = root

# Create the leaves
a = BTreeNode()
a.keys = [10]

b = BTreeNode()
b.keys = [25]

c = BTreeNode()
c.keys = [40]

node.children = [a]
right.children = [b, c]

a.parent = node
for child in right.children:
    child.parent = right

parent = tree._merge(node)

assert parent is root
assert root.keys == []
assert root.children == [node]
assert node.keys == [20, 30]
assert node.children == [a, b, c]

for child in node.children:
    assert child.parent is node

print("Reto 8: Todos los tests superados correctamente.")

# 🏆 Challenge 9 · Fix an underflow after deletion

🎯 **Goal:** Implement `BTree._fix_underflow(node)` to restore B-tree properties when a node has fewer than the required minimum number of keys.

The method must:

1. **Check for underflow.** Except for the root, each node must have at least `ceil(m / 2) - 1` keys.
2. **Attempt redistribution:** if a sibling can lend a key, call `_borrow_from_left()` or `_borrow_from_right()`, preferring the left sibling.
3. **Merge if needed:** if neither sibling can lend, call `_merge(node)`.
4. **Propagate repairs upward:** after merging, check whether the parent is also deficient and continue if necessary.
5. **Update the root:** if the root has no keys, replace it with its only child, or set `self.root = None` if the tree is empty.

**Important:** This method does not delete keys; it only fixes deficiencies that arise during deletion.

🕒 **Estimated time:** 25 minutes  
🔴 **Difficulty:** Advanced

### 💡 Hint

Use a `while` loop to repair underflows that propagate toward the root.

At each iteration:

- Compute the minimum number of keys.
- First check whether the left sibling can lend, then the right sibling.
- If neither can lend, merge and continue checking the parent.

You can reuse `_borrow_from_left()`, `_borrow_from_right()`, and `_merge()` from previous challenges.


This cell tests Challenge 9 with manually constructed trees, so it does not depend on `insert()` or `remove()`.

The tests cover redistributions, merges, upward propagation of underflow, and reduction in tree height.


In [ ]:
# Tests Challenge 9: BTree._fix_underflow()

# Test 1: Redistribution from the left sibling
tree = BTree(4)

root = BTreeNode(is_leaf=False)
root.keys = [20]
tree.root = root

left = BTreeNode()
left.keys = [5, 10]

node = BTreeNode()
node.keys = []  # Nodo deficitario

root.children = [left, node]
left.parent = root
node.parent = root

tree._fix_underflow(node)

assert root.keys == [10]
assert left.keys == [5]
assert node.keys == [20]
assert root.children == [left, node]


# Test 2: Redistribution from the right sibling
tree = BTree(4)

root = BTreeNode(is_leaf=False)
root.keys = [20]
tree.root = root

node = BTreeNode()
node.keys = []  # Nodo deficitario

right = BTreeNode()
right.keys = [30, 40]

root.children = [node, right]
node.parent = root
right.parent = root

tree._fix_underflow(node)

assert root.keys == [30]
assert node.keys == [20]
assert right.keys == [40]


# Test 3: Prefer the left sibling
tree = BTree(4)

root = BTreeNode(is_leaf=False)
root.keys = [20, 40]
tree.root = root

left = BTreeNode()
left.keys = [5, 10]

node = BTreeNode()
node.keys = []  # Nodo deficitario

right = BTreeNode()
right.keys = [50, 60]

root.children = [left, node, right]
for child in root.children:
    child.parent = root

tree._fix_underflow(node)

assert root.keys == [10, 40]
assert left.keys == [5]
assert node.keys == [20]
assert right.keys == [50, 60]


# Test 4: Merge without upward propagation
tree = BTree(4)

root = BTreeNode(is_leaf=False)
root.keys = [10, 20]
tree.root = root

left = BTreeNode()
left.keys = [5]

node = BTreeNode()
node.keys = []  # Nodo deficitario

right = BTreeNode()
right.keys = [25]

root.children = [left, node, right]
for child in root.children:
    child.parent = root

tree._fix_underflow(node)

assert root.keys == [20]
assert root.children == [left, right]
assert left.keys == [5, 10]
assert right.keys == [25]
assert tree.root is root


# Test 5: Merge that reduces tree height
tree = BTree(4)

root = BTreeNode(is_leaf=False)
root.keys = [20]
tree.root = root

node = BTreeNode()
node.keys = []  # Nodo deficitario

right = BTreeNode()
right.keys = [30]

root.children = [node, right]
node.parent = root
right.parent = root

tree._fix_underflow(node)

assert tree.root is node
assert tree.root.keys == [20, 30]
assert tree.root.is_leaf
assert tree.root.parent is None


# Test 6: Underflow propagation to the root
tree = BTree(4)

root = BTreeNode(is_leaf=False)
root.keys = [30]
tree.root = root

left = BTreeNode(is_leaf=False)
left.keys = [10]

right = BTreeNode(is_leaf=False)
right.keys = [40]

root.children = [left, right]
left.parent = root
right.parent = root

a = BTreeNode()
a.keys = [5]

node = BTreeNode()
node.keys = []  # Nodo deficitario

b = BTreeNode()
b.keys = [35]

c = BTreeNode()
c.keys = [45]

left.children = [a, node]
right.children = [b, c]

for child in left.children:
    child.parent = left
for child in right.children:
    child.parent = right

tree._fix_underflow(node)

# First merge: [5] + 10 + [] -> [5, 10]
# Second merge: [40] + 30 + [10] -> [10, 30, 40]
# The original root disappears.

assert tree.root is left
assert tree.root.keys == [30, 40]
assert tree.root.children == [a, b, c]
assert a.keys == [5, 10]
assert b.keys == [35]
assert c.keys == [45]
assert tree.root.parent is None

for child in tree.root.children:
    assert child.parent is tree.root


# Test 7: The tree becomes empty
tree = BTree(4)

root = BTreeNode()
root.keys = []
tree.root = root

tree._fix_underflow(root)

assert tree.root is None


# Test 8: Node without underflow (must remain unchanged)
tree = BTree(4)

root = BTreeNode(is_leaf=False)
root.keys = [20]
tree.root = root

left = BTreeNode()
left.keys = [10]

right = BTreeNode()
right.keys = [30]

root.children = [left, right]
left.parent = root
right.parent = root

tree._fix_underflow(left)

assert tree.root is root
assert root.keys == [20]
assert left.keys == [10]
assert right.keys == [30]
assert root.children == [left, right]

print("Reto 9: Todos los tests superados correctamente.")

# 🏆 Challenge 10 · Delete a key from a B-tree

🎯 **Goal:** Implement `BTree.remove(key)` to delete a key while preserving B-tree properties.

The method must:

1. **Find the key** using `search(key)`. Return `False` if it is absent.
2. **If the key is in a leaf**, remove it directly.
3. **If the key is in an internal node**, replace it with its predecessor (the largest key in the left subtree) using `_get_predecessor(node, i)`. Then delete that predecessor from its leaf.
4. **Fix any underflow** by calling `_fix_underflow()` on the leaf from which the key was removed.
5. **Return `True`** after a successful deletion.

**Important:** Even when the key is in an internal node, the actual removal is always performed at a leaf.

🕒 **Estimated time:** 20 minutes  
🔴 **Difficulty:** Advanced

### 💡 Hint

Reuse methods from the previous challenges:

- `search(key)`: locates the key and returns `(node, i)`.
- `_get_predecessor(node, i)`: finds a key's predecessor and its leaf.
- `_fix_underflow(node)`: fixes any underflow after deletion.

Remember that `remove()` must return a Boolean indicating whether deletion succeeded.


This cell contains assertions to test Challenge 10 with several deletion cases.

The trees are built manually so the tests do not depend on `insert()`. Your implementations from Challenges 7–9 must work correctly because `remove()` relies on them.


In [ ]:
# Tests Challenge 10: BTree.remove()

# Test 1: Delete a missing key
tree = BTree(4)

root = BTreeNode()
root.keys = [10, 20]
tree.root = root

assert tree.remove(30) is False
assert root.keys == [10, 20]


# Test 2: Delete a leaf key without underflow
tree = BTree(4)

root = BTreeNode()
root.keys = [10, 20, 30]
tree.root = root

assert tree.remove(20) is True
assert root.keys == [10, 30]
assert tree.root is root


# Test 3: Delete a leaf key requiring redistribution
tree = BTree(4)

root = BTreeNode(is_leaf=False)
root.keys = [20]
tree.root = root

left = BTreeNode()
left.keys = [5, 10]

right = BTreeNode()
right.keys = [30]

root.children = [left, right]
left.parent = root
right.parent = root

assert tree.remove(30) is True

assert root.keys == [10]
assert left.keys == [5]
assert right.keys == [20]
assert tree.search(30) is None


# Test 4: Delete a leaf key requiring a merge
tree = BTree(4)

root = BTreeNode(is_leaf=False)
root.keys = [20]
tree.root = root

left = BTreeNode()
left.keys = [10]

right = BTreeNode()
right.keys = [30]

root.children = [left, right]
left.parent = root
right.parent = root

assert tree.remove(30) is True

assert tree.root is left
assert tree.root.keys == [10, 20]
assert tree.root.parent is None
assert tree.search(30) is None


# Test 5: Delete a key from an internal node
tree = BTree(4)

root = BTreeNode(is_leaf=False)
root.keys = [20]
tree.root = root

left = BTreeNode()
left.keys = [5, 10]

right = BTreeNode()
right.keys = [30, 40]

root.children = [left, right]
left.parent = root
right.parent = root

assert tree.remove(20) is True

# Replace 20 with its predecessor (10)
assert root.keys == [10]
assert left.keys == [5]
assert right.keys == [30, 40]
assert tree.search(20) is None


# Test 6: Delete an internal key that triggers a merge
tree = BTree(4)

root = BTreeNode(is_leaf=False)
root.keys = [20]
tree.root = root

left = BTreeNode()
left.keys = [10]

right = BTreeNode()
right.keys = [30]

root.children = [left, right]
left.parent = root
right.parent = root

assert tree.remove(20) is True

# The predecessor (10) replaces 20.
# Removing it from the leaf triggers a merge.
assert tree.root is left
assert tree.root.keys == [10, 30]
assert tree.root.parent is None
assert tree.search(20) is None


# Test 7: Delete the tree's only key
tree = BTree(4)

root = BTreeNode()
root.keys = [10]
tree.root = root

assert tree.remove(10) is True
assert tree.root is None
assert tree.remove(10) is False


# Test 8: Consecutive deletions
tree = BTree(4)

root = BTreeNode(is_leaf=False)
root.keys = [20]
tree.root = root

left = BTreeNode()
left.keys = [5, 10]

right = BTreeNode()
right.keys = [30, 40]

root.children = [left, right]
left.parent = root
right.parent = root

for key in [5, 10, 20, 30, 40]:
    assert tree.remove(key) is True
    assert tree.search(key) is None

assert tree.root is None


# Test 9: Underflow propagation to the root
tree = BTree(4)

root = BTreeNode(is_leaf=False)
root.keys = [30]
tree.root = root

left = BTreeNode(is_leaf=False)
left.keys = [10]

right = BTreeNode(is_leaf=False)
right.keys = [40]

root.children = [left, right]
left.parent = root
right.parent = root

a = BTreeNode()
a.keys = [5]

b = BTreeNode()
b.keys = [20]

c = BTreeNode()
c.keys = [35]

d = BTreeNode()
d.keys = [45]

left.children = [a, b]
right.children = [c, d]

for child in left.children:
    child.parent = left
for child in right.children:
    child.parent = right

assert tree.remove(20) is True

# The deletion causes two merges and reduces tree height.
assert tree.root is left
assert tree.root.keys == [30, 40]
assert tree.root.children == [a, c, d]
assert a.keys == [5, 10]
assert c.keys == [35]
assert d.keys == [45]
assert tree.root.parent is None

for child in tree.root.children:
    assert child.parent is tree.root

assert tree.search(20) is None


print("Reto 10: Todos los tests superados correctamente.")

# 🏆 Challenge 11 · Diagram exercise: insertion and splitting a leaf

🎯 **Goal:** Draw an insertion that causes a `split`.

Consider this order-4 B-tree:

```text
          [10 | 20]
        /     |      \
      [5] [12|15|18] [25|30]
```

Insert key **17**.

### ✏️ In your solution, draw
1. The leaf after temporarily inserting 17 in sorted order.
2. The resulting leaf split.
3. The key promoted to the parent.
4. The final tree.

🕒 **Estimated time:** 10 minutes  
🟢 **Difficulty:** Easy


# 🏆 Challenge 12 · Diagram exercise: deletion, merging, and propagation to the root

🎯 **Goal:** Solve a deletion in which merging makes the **parent deficient too**, so repairs must propagate one level upward.

Order-4 B-tree:

```text
                         [50]
                    /            \
                  [20]            [70]
                /      \        /      \
              [10]     [30]   [60]   [80 | 90]
```

Delete key **10**.

### ✏️ Draw and explain

1. The state immediately after deleting 10.
2. The necessary merge at the lower level.
3. Why the internal node `[20]` becomes deficient.
4. How to rebalance the upper level.
5. The final tree.

🕒 **Estimated time:** 15 minutes  
🟡 **Difficulty:** Intermediate


# 🏆 Challenge 13 · Diagram exercise: borrowing at the upper level after a merge

🎯 **Goal:** Solve a deletion where the first repair is a **merge**, but the resulting deficiency in the parent is fixed by **borrowing/rotation**.

Order-4 B-tree:

```text
                           [50]
                     /              \
                   [20]            [70 | 90]
                 /      \         /    |     \
               [10]     [30]    [60] [80] [100]
```

Delete key **10**.

### ✏️ Draw and explain

1. Deleting 10.
2. The merge at leaf level.
3. The underflow in the left internal node.
4. Borrowing from the right internal sibling through the root.
5. The final tree.

🕒 **Estimated time:** 18 minutes  
🔴 **Difficulty:** Advanced


# 🏆 Challenge 14 · Diagram exercise: internal key and upward propagation

🎯 **Goal:** Combine deletion of an **internal key** with rebalancing that propagates upward.

Order-4 B-tree:

```text
                         [50]
                    /            \
                  [20]            [70]
                /      \        /      \
              [10]     [30]   [60]   [80 | 90]
```

Delete the internal key **20** using its **immediate successor**.

### ✏️ Draw and explain

1. Find the successor of 20.
2. Replace 20 with its successor.
3. Delete that successor from its leaf.
4. Fix the underflow in that leaf.
5. Continue rebalancing if the parent becomes deficient.
6. Draw the final tree.

🕒 **Estimated time:** 20 minutes  
🔴 **Difficulty:** Advanced


## 🌳 Keep practicing B-trees!

Congratulations! 🎉 You have completed the B-tree implementation challenges.

To reinforce what you have learned, practice with the online [Tree Lab visualizer](https://treelab.pranayboreddy.space/) we used in class.

Try inserting and deleting keys. Watch how splits, redistributions, and merges restructure the tree.

**Does the result match what you would obtain by applying the algorithms you implemented?**
